# Orientation Deep Dive

This notebook explores how pose detection performs across portrait and landscape orientations.

## Key Questions
1. Does MediaPipe work equally well on horizontal body poses?
2. How do normalized coordinates differ between orientations?
3. Should we rotate landscape frames to portrait before inference?
4. What's the impact on our normalization pipeline?

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from pathlib import Path
import sys

# Add parent path for imports
sys.path.append(str(Path.cwd().parent.parent / 'src'))

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

print("Setup complete")

## 1. Load Test Videos

Load one portrait (e.g., squat/lunge) and one landscape (e.g., pushup/plank) video.

In [ ]:
# TODO: Update these paths to your test videos
PORTRAIT_VIDEO = "../../data/raw_videos/lunge/portrait_side_good_001.mp4"
LANDSCAPE_VIDEO = "../../data/raw_videos/pushup/landscape_side_good_001.mp4"

def load_sample_frames(video_path, n_frames=5):
    """Load evenly-spaced sample frames from video."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    frames = []
    
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    cap.release()
    return frames

# Load if files exist
if Path(PORTRAIT_VIDEO).exists():
    portrait_frames = load_sample_frames(PORTRAIT_VIDEO)
    print(f"Loaded {len(portrait_frames)} portrait frames")
else:
    print(f"Portrait video not found: {PORTRAIT_VIDEO}")
    portrait_frames = []

if Path(LANDSCAPE_VIDEO).exists():
    landscape_frames = load_sample_frames(LANDSCAPE_VIDEO)
    print(f"Loaded {len(landscape_frames)} landscape frames")
else:
    print(f"Landscape video not found: {LANDSCAPE_VIDEO}")
    landscape_frames = []

## 2. Detect Poses and Visualize

In [ ]:
def detect_and_visualize(frames, title_prefix):
    """Detect poses and visualize with skeleton overlay."""
    if not frames:
        print("No frames to process")
        return []
    
    results = []
    
    with mp_pose.Pose(
        model_complexity=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 6))
        if len(frames) == 1:
            axes = [axes]
        
        for i, frame in enumerate(frames):
            result = pose.process(frame)
            
            # Draw on copy
            annotated = frame.copy()
            
            if result.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated,
                    result.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS
                )
                
                # Extract visibility stats
                vis = [lm.visibility for lm in result.pose_landmarks.landmark]
                avg_vis = np.mean(vis)
                results.append({
                    'frame_idx': i,
                    'detected': True,
                    'avg_visibility': avg_vis,
                    'landmarks': result.pose_landmarks
                })
                title_suffix = f"Vis: {avg_vis:.2f}"
            else:
                results.append({'frame_idx': i, 'detected': False})
                title_suffix = "No detection"
            
            axes[i].imshow(annotated)
            axes[i].set_title(f"{title_prefix} #{i+1}\n{title_suffix}")
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return results

print("Portrait Poses:")
portrait_results = detect_and_visualize(portrait_frames, "Portrait")

print("\nLandscape Poses:")
landscape_results = detect_and_visualize(landscape_frames, "Landscape")

## 3. Analyze Coordinate Distributions

Compare how landmarks distribute in normalized coordinate space.

In [ ]:
def extract_landmark_coords(results):
    """Extract x, y coordinates from detection results."""
    all_x, all_y = [], []
    
    for r in results:
        if r.get('detected') and r.get('landmarks'):
            for lm in r['landmarks'].landmark:
                all_x.append(lm.x)
                all_y.append(lm.y)
    
    return np.array(all_x), np.array(all_y)

p_x, p_y = extract_landmark_coords(portrait_results)
l_x, l_y = extract_landmark_coords(landscape_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if len(p_x) > 0:
    axes[0].scatter(p_x, p_y, alpha=0.5, c='green', label='Portrait')
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(1, 0)  # Flip Y to match image coordinates
    axes[0].set_xlabel('X (normalized)')
    axes[0].set_ylabel('Y (normalized)')
    axes[0].set_title(f'Portrait Landmark Distribution\n(n={len(p_x)} points)')
    axes[0].set_aspect('equal')

if len(l_x) > 0:
    axes[1].scatter(l_x, l_y, alpha=0.5, c='blue', label='Landscape')
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(1, 0)
    axes[1].set_xlabel('X (normalized)')
    axes[1].set_ylabel('Y (normalized)')
    axes[1].set_title(f'Landscape Landmark Distribution\n(n={len(l_x)} points)')
    axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

# Summary stats
print("\n--- Coordinate Statistics ---")
if len(p_x) > 0:
    print(f"Portrait X: mean={p_x.mean():.3f}, std={p_x.std():.3f}, range=[{p_x.min():.3f}, {p_x.max():.3f}]")
    print(f"Portrait Y: mean={p_y.mean():.3f}, std={p_y.std():.3f}, range=[{p_y.min():.3f}, {p_y.max():.3f}]")
if len(l_x) > 0:
    print(f"Landscape X: mean={l_x.mean():.3f}, std={l_x.std():.3f}, range=[{l_x.min():.3f}, {l_x.max():.3f}]")
    print(f"Landscape Y: mean={l_y.mean():.3f}, std={l_y.std():.3f}, range=[{l_y.min():.3f}, {l_y.max():.3f}]")

## 4. Test Rotation Approach

Does rotating landscape frames to portrait improve detection?

In [ ]:
def rotate_frame_90(frame):
    """Rotate frame 90 degrees clockwise."""
    return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

def compare_rotation_impact(frames, title):
    """Compare detection with and without rotation."""
    if not frames:
        return
    
    original_vis = []
    rotated_vis = []
    
    with mp_pose.Pose(
        model_complexity=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        for frame in frames:
            # Original
            result = pose.process(frame)
            if result.pose_landmarks:
                vis = np.mean([lm.visibility for lm in result.pose_landmarks.landmark])
                original_vis.append(vis)
            else:
                original_vis.append(0)
            
            # Rotated
            rotated = rotate_frame_90(frame)
            result = pose.process(rotated)
            if result.pose_landmarks:
                vis = np.mean([lm.visibility for lm in result.pose_landmarks.landmark])
                rotated_vis.append(vis)
            else:
                rotated_vis.append(0)
    
    print(f"\n{title}:")
    print(f"  Original avg visibility: {np.mean(original_vis):.3f}")
    print(f"  Rotated avg visibility: {np.mean(rotated_vis):.3f}")
    print(f"  Improvement: {np.mean(rotated_vis) - np.mean(original_vis):+.3f}")
    
    return original_vis, rotated_vis

print("Testing rotation impact...")
compare_rotation_impact(portrait_frames, "Portrait (should NOT benefit)")
compare_rotation_impact(landscape_frames, "Landscape (might benefit)")

## 5. Normalization Consistency Check

Verify our hip-center, torso-scale normalization works for both orientations.

In [ ]:
# Indices
L_HIP, R_HIP = 23, 24
L_SHOULDER, R_SHOULDER = 11, 12

def normalize_pose(landmarks):
    """Apply hip-center, torso-scale normalization."""
    lm = landmarks.landmark
    
    # Extract coordinates
    coords = np.array([[l.x, l.y, l.z] for l in lm])
    
    # Hip center
    hip_center = (coords[L_HIP] + coords[R_HIP]) / 2
    
    # Shoulder center
    shoulder_center = (coords[L_SHOULDER] + coords[R_SHOULDER]) / 2
    
    # Torso scale (distance from hip to shoulder center)
    torso_vec = shoulder_center - hip_center
    torso_scale = np.sqrt(torso_vec[0]**2 + torso_vec[1]**2)
    
    if torso_scale < 0.01:  # Prevent division by zero
        torso_scale = 0.01
    
    # Normalize
    normalized = (coords - hip_center) / torso_scale
    
    return normalized, hip_center, torso_scale

def plot_normalized_poses(results_list, labels):
    """Plot normalized poses for comparison."""
    fig, axes = plt.subplots(1, len(results_list), figsize=(6 * len(results_list), 6))
    if len(results_list) == 1:
        axes = [axes]
    
    for ax, results, label in zip(axes, results_list, labels):
        all_normalized = []
        
        for r in results:
            if r.get('detected') and r.get('landmarks'):
                normalized, _, _ = normalize_pose(r['landmarks'])
                all_normalized.append(normalized)
        
        if all_normalized:
            # Plot all poses overlaid
            for norm in all_normalized:
                ax.scatter(norm[:, 0], norm[:, 1], alpha=0.3, s=20)
            
            # Mean pose
            mean_pose = np.mean(all_normalized, axis=0)
            ax.scatter(mean_pose[:, 0], mean_pose[:, 1], c='red', s=50, marker='x', label='Mean')
        
        ax.set_xlabel('X (normalized)')
        ax.set_ylabel('Y (normalized)')
        ax.set_title(f'{label}\nNormalized Coordinates')
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_aspect('equal')
        ax.legend()
    
    plt.tight_layout()
    plt.show()

plot_normalized_poses(
    [portrait_results, landscape_results],
    ['Portrait (Lunge)', 'Landscape (Push-up)']
)

## 6. Conclusions

Fill in your findings here:

### Detection Quality
- Portrait detection rate: ___%
- Landscape detection rate: ___%
- Problem joints in landscape: ___

### Rotation Impact
- Does rotation help landscape detection? ___
- Trade-offs: ___

### Normalization
- Does hip-center normalization work for both? ___
- Coordinate range differences: ___

### Recommendations
1. ___
2. ___
3. ___